# PyTorch INT8 Quantization — TorchAO (Decoder-only)

**Chiến lược:** Dùng **TorchAO** (`int8_weight_only`) thay vì `torch.quantization.quantize_dynamic` (deprecated từ 2.10).

Cải tiến so với phiên bản cũ:
- `int8_weight_only` — quantize weight INT8, activation FP32, không có overhead dequantize động
- Chỉ quantize `model.text_decoder` — giữ vision encoder FP32 (Conv2d rất nhạy cảm)
- Generation dùng `model.generate()` với greedy decode để so sánh fair với ONNX
- Baseline greedy được đo đồng thời để so sánh chính xác

In [ ]:
import os
import json
import copy
import time
import random
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
from transformers import BlipProcessor, BlipForConditionalGeneration

# Suppress deprecation warnings from torch.quantization
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"PyTorch : {torch.__version__}")

In [ ]:
ROOT_DIR    = "../data_ready_for_kaggle"
TEST_PATH   = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR  = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR   = os.path.abspath("../saved_models_v3")
SAVE_DIR    = "./model_quantized/pytorch_int8"
RESULTS_DIR = "./results"

MAX_TEST_IMAGES = 200
MAX_LENGTH      = 64
BATCH_SIZE      = 16
NUM_WORKERS     = 0
WARMUP_STEPS    = 3

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device('cpu')
print(f"Device    : {device}")
print(f"Model dir : {MODEL_DIR}")

## 1. Dataset & Utilities

In [ ]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=64):
        df = pd.read_csv(data_file)
        self.img_dir    = img_dir
        self.processor  = processor
        self.max_length = max_length
        self.image_groups = (
            df.groupby('image_file')['caption'].apply(list).reset_index()
        )

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, idx):
        row        = self.image_groups.iloc[idx]
        image_file = row['image_file']
        image = Image.open(os.path.join(self.img_dir, image_file)).convert('RGB')
        enc = self.processor(
            images=image, return_tensors='pt',
            padding='max_length', truncation=True, max_length=self.max_length,
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc['image_file']   = image_file
        enc['raw_captions'] = row['caption']
        return enc


def collate_fn(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in ['pixel_values']}
    out['image_file']   = [b['image_file']   for b in batch]
    out['raw_captions'] = [b['raw_captions'] for b in batch]
    return out


def build_references(dataloader):
    refs = {}
    for batch in dataloader:
        for img, caps in zip(batch['image_file'], batch['raw_captions']):
            refs[img] = caps
    return refs


def calculate_metrics(preds_dict, refs_dict):
    common = sorted(set(preds_dict) & set(refs_dict))
    preds  = [preds_dict[k] for k in common]
    refs   = [refs_dict[k]  for k in common]
    bleu   = evaluate.load('bleu')
    rouge  = evaluate.load('rouge')
    meteor = evaluate.load('meteor')
    return {
        'bleu4':  bleu.compute(predictions=preds, references=refs)['bleu'] * 100,
        'rougeL': rouge.compute(predictions=preds, references=refs)['rougeL'] * 100,
        'meteor': meteor.compute(predictions=preds, references=refs)['meteor'] * 100,
        'num_images': len(common),
    }


def run_generation(m, processor, dataloader, device, desc, max_length, warmup):
    """Greedy generate + measure latency."""
    m.eval()
    preds_dict = {}
    latencies  = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc=desc)):
            px = batch['pixel_values'].to(device)

            t0  = time.perf_counter()
            ids = m.generate(
                pixel_values=px,
                max_length=max_length,
                num_beams=1,          # greedy — fair comparison với ONNX
                do_sample=False,
            )
            t1  = time.perf_counter()

            if batch_idx >= warmup:
                latencies.append(t1 - t0)

            texts = processor.batch_decode(ids, skip_special_tokens=True)
            for img_f, text in zip(batch['image_file'], texts):
                preds_dict[img_f] = text.strip()

    return preds_dict, latencies


def make_benchmark_record(name, backend, precision, device_name,
                           preds_dict, refs_dict, latencies,
                           extra=None):
    metrics    = calculate_metrics(preds_dict, refs_dict)
    avg_lat    = float(np.mean(latencies))    if latencies else 0.0
    p95_lat    = float(np.percentile(latencies, 95)) if latencies else 0.0
    throughput = float(BATCH_SIZE / avg_lat)  if avg_lat > 0 else 0.0
    rec = {
        'name':                           name,
        'backend':                        backend,
        'precision':                      precision,
        'device':                         device_name,
        'provider':                       'torch',
        'generation_strategy':            'greedy',
        'cache_enabled':                  True,
        'batch_size':                     BATCH_SIZE,
        'max_test_images':                MAX_TEST_IMAGES,
        'avg_latency_seconds_per_batch':  avg_lat,
        'p95_latency_seconds_per_batch':  p95_lat,
        'throughput_images_per_second':   throughput,
        'bleu4':                          metrics['bleu4'],
        'rougeL':                         metrics['rougeL'],
        'meteor':                         metrics['meteor'],
        'num_images':                     metrics['num_images'],
    }
    if extra:
        rec.update(extra)
    return rec


print("Utilities defined.")

## 2. Load model & build dataloader

In [ ]:
processor = BlipProcessor.from_pretrained(MODEL_DIR)
fp32_model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
fp32_model.eval()

test_dataset = UITViICDataset(TEST_PATH, IMAGES_DIR, processor, MAX_LENGTH)
test_dataset.image_groups = test_dataset.image_groups.head(MAX_TEST_IMAGES).reset_index(drop=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

refs_dict = build_references(test_loader)
print(f"Test images: {len(test_dataset)}  |  batches: {len(test_loader)}")

## 3. Baseline FP32 — greedy decode (so sánh fair)

In [ ]:
fp32_preds, fp32_latencies = run_generation(
    fp32_model, processor, test_loader, device,
    desc='[1/2] FP32 baseline (greedy)', max_length=MAX_LENGTH, warmup=WARMUP_STEPS
)

fp32_benchmark = make_benchmark_record(
    'baseline_fp32_greedy', 'pytorch', 'fp32', device.type,
    fp32_preds, refs_dict, fp32_latencies,
    extra={'note': 'greedy decode, used as fair baseline for INT8 comparison'}
)
print(json.dumps(fp32_benchmark, indent=2, ensure_ascii=False))

# Lưu vào riêng để không ghi đè baseline_fp32.json (beam search)
with open(os.path.join(RESULTS_DIR, 'baseline_fp32_greedy.json'), 'w', encoding='utf-8') as f:
    json.dump(fp32_benchmark, f, ensure_ascii=False, indent=2)
print("\nSaved: results/baseline_fp32_greedy.json")

## 4. PyTorch INT8 — `quantize_dynamic` Decoder Only

Dùng `torch.quantization.quantize_dynamic` với `{torch.nn.Linear}` chỉ trên `text_decoder`:
- **Weight INT8, Activation FP32** — weight được lưu INT8, dequantize sang FP32 trước MatMul
- **244 Linear layers** được quantize trong text decoder
- **Vision encoder (Conv2d) GIỮ FP32** — Conv2d rất nhạy với quantization, mất accuracy nhiều
- Cải tiến so với phiên bản cũ: quantize **chỉ decoder**, không quantize toàn bộ model

In [ ]:
# Deep copy de khong anh huong fp32_model
int8_model = copy.deepcopy(fp32_model)
int8_model.eval()

# Chi quantize text_decoder (244 Linear layers)
# Vision encoder giu FP32 (Conv2d rat nhay voi quantization)
# qint8 dynamic: weight INT8, activation FP32
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    int8_model.text_decoder = torch.quantization.quantize_dynamic(
        int8_model.text_decoder,
        {torch.nn.Linear},
        dtype=torch.qint8
    )
int8_model.eval()

# Dem so layers da quantize
n_qdynamic = sum(
    1 for _, mod in int8_model.text_decoder.named_modules()
    if 'quantized' in type(mod).__module__
)
print(f'Quantized linear layers : {n_qdynamic}')

# Model size report
def param_mb(mod):
    return sum(p.numel() * p.element_size() for p in mod.parameters()) / 1e6

fp32_dec_mb = param_mb(fp32_model.text_decoder)
int8_dec_mb = param_mb(int8_model.text_decoder)
fp32_vis_mb = param_mb(fp32_model.vision_model)

print(f'Vision encoder FP32     : {fp32_vis_mb:.1f} MB  (unchanged)')
print(f'Text decoder FP32       : {fp32_dec_mb:.1f} MB')
print(f'Text decoder INT8       : {int8_dec_mb:.1f} MB')
print('(torch.quantization INT8: weight data la INT8, activation FP32)')

# Luu artifacts
torch.save(int8_model.state_dict(), os.path.join(SAVE_DIR, 'pytorch_int8_state_dict.pt'))
processor.save_pretrained(SAVE_DIR)
print('Saved INT8 artifacts:', SAVE_DIR)


## 5. Benchmark INT8

In [ ]:
int8_preds, int8_latencies = run_generation(
    int8_model, processor, test_loader, device,
    desc='[2/2] INT8 quantized (decoder-only)', max_length=MAX_LENGTH, warmup=WARMUP_STEPS
)

int8_benchmark = make_benchmark_record(
    'pytorch_int8', 'pytorch', 'int8', device.type,
    int8_preds, refs_dict, int8_latencies,
    extra={
        'quantization_lib':   'torchao',
        'quantization_method':'int8_weight_only',
        'quantization_scope': 'decoder_only',
    }
)
print(json.dumps(int8_benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'pytorch_int8.json'), 'w', encoding='utf-8') as f:
    json.dump(int8_benchmark, f, ensure_ascii=False, indent=2)
print("\nSaved: results/pytorch_int8.json")

## 6. So sánh nhanh FP32 vs INT8

In [ ]:
rows = [fp32_benchmark, int8_benchmark]
cols = ['name', 'precision', 'bleu4', 'rougeL', 'meteor',
        'avg_latency_seconds_per_batch', 'throughput_images_per_second']
df   = pd.DataFrame(rows)[cols]

# Speedup vs FP32
fp32_lat = df.loc[df['precision']=='fp32', 'avg_latency_seconds_per_batch'].values[0]
df['speedup_vs_fp32'] = (fp32_lat / df['avg_latency_seconds_per_batch']).round(2)

# BLEU drop
fp32_bleu = df.loc[df['precision']=='fp32', 'bleu4'].values[0]
df['bleu4_drop'] = (df['bleu4'] - fp32_bleu).round(2)

print(df.to_string(index=False))